In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    f1_score
)

from xgboost import XGBClassifier

In [2]:
ARTIFACT_DIR = Path("artifacts/cleaning_only")

meta = joblib.load(ARTIFACT_DIR / "meta.joblib")
cleaned_outputs = joblib.load(ARTIFACT_DIR / "cleaned_outputs.joblib")

# Optional: load cleaning pipelines (may require same class definitions in environment)
try:
    cleaning_pipelines = joblib.load(ARTIFACT_DIR / "cleaning_pipelines.joblib")
    print("Loaded cleaning_pipelines.joblib")
except Exception as e:
    cleaning_pipelines = None
    print("Skipping cleaning_pipelines.joblib load:", e)

print("Meta:", meta)

Skipping cleaning_pipelines.joblib load: Can't get attribute 'ParseDatesTransformer' on <module '__main__'>
Meta: {'snapshot_date': '2018-12-31', 'horizon_months': 12, 'split_years': {'train_end_year': 2015, 'val_year': 2016, 'test_year': 2017}, 'fundamental_drop': ['grade', 'sub_grade', 'int_rate', 'installment'], 'non_default_statuses': ['Does not meet the credit policy. Status: Fully Paid', 'Fully Paid'], 'default_statuses': ['Charged Off', 'Default', 'Does not meet the credit policy. Status: Charged Off']}


In [3]:
#  Unpack cleaned datasets + labels
X_train_fund_clean = cleaned_outputs["X_train_fund_clean"]
X_val_fund_clean = cleaned_outputs["X_val_fund_clean"]
X_test_fund_clean = cleaned_outputs["X_test_fund_clean"]

X_train_fund_xgb = cleaned_outputs["X_train_fund_xgb"]
X_val_fund_xgb = cleaned_outputs["X_val_fund_xgb"]
X_test_fund_xgb = cleaned_outputs["X_test_fund_xgb"]

X_train_full_xgb = cleaned_outputs["X_train_full_xgb"]
X_val_full_xgb = cleaned_outputs["X_val_full_xgb"]
X_test_full_xgb = cleaned_outputs["X_test_full_xgb"]

y_train_f = cleaned_outputs["y_train_f"]
y_val_f = cleaned_outputs["y_val_f"]
y_test_f = cleaned_outputs["y_test_f"]

y_train_full = cleaned_outputs["y_train_full"]
y_val_full = cleaned_outputs["y_val_full"]
y_test_full = cleaned_outputs["y_test_full"]

print("Shapes:")
print("  Fund logistic:", X_train_fund_clean.shape, X_val_fund_clean.shape, X_test_fund_clean.shape)
print("  Fund XGB:", X_train_fund_xgb.shape, X_val_fund_xgb.shape, X_test_fund_xgb.shape)
print("  Full XGB:", X_train_full_xgb.shape, X_val_full_xgb.shape, X_test_full_xgb.shape)
print("  Labels:", y_train_f.shape, y_val_f.shape, y_test_f.shape)

Shapes:
  Fund logistic: (828685, 44) (292588, 44) (168721, 44)
  Fund XGB: (828685, 44) (292588, 44) (168721, 44)
  Full XGB: (828685, 52) (292588, 52) (168721, 52)
  Labels: (828685,) (292588,) (168721,)


In [4]:
#  Categorical encoder utility (OHE)
def encode_categoricals(X_train, X_val, X_test, scale_numeric: bool):
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()

    cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    # normalize categorical missing values to np.nan for OHE compatibility
    for d in (X_train, X_val, X_test):
        for c in cat_cols:
            col = d[c].astype("object")
            d[c] = col.where(pd.notna(col), np.nan)

    if scale_numeric:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    encoder = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop"
    )

    X_train_enc = encoder.fit_transform(X_train)
    X_val_enc = encoder.transform(X_val)
    X_test_enc = encoder.transform(X_test)

    return encoder, X_train_enc, X_val_enc, X_test_enc, num_cols, cat_cols

In [ ]:
# Encode the 3 model datasets
# Logistic (fundamental): scale numerics
enc_fund_log, X_train_fund_log_enc, X_val_fund_log_enc, X_test_fund_log_enc, num_f_log, cat_f_log = encode_categoricals(
    X_train_fund_clean, X_val_fund_clean, X_test_fund_clean, scale_numeric=True
)

# XGB (fundamental): no scaling needed
enc_fund_xgb, X_train_fund_xgb_enc, X_val_fund_xgb_enc, X_test_fund_xgb_enc, num_f_xgb, cat_f_xgb = encode_categoricals(
    X_train_fund_xgb, X_val_fund_xgb, X_test_fund_xgb, scale_numeric=False
)

# XGB (full): no scaling needed
enc_full_xgb, X_train_full_xgb_enc, X_val_full_xgb_enc, X_test_full_xgb_enc, num_full_xgb, cat_full_xgb = encode_categoricals(
    X_train_full_xgb, X_val_full_xgb, X_test_full_xgb, scale_numeric=False
)

print("fund_log cats:", len(cat_f_log), "shape:", X_train_fund_log_enc.shape)
print("fund_xgb cats:", len(cat_f_xgb), "shape:", X_train_fund_xgb_enc.shape)
print("full_xgb cats:", len(cat_full_xgb), "shape:", X_train_full_xgb_enc.shape)

fund_log cats: 5 shape: (828685, 96)
fund_xgb cats: 5 shape: (828685, 96)
full_xgb cats: 7 shape: (828685, 135)


In [6]:
# Train the 3 models
# class imbalance helper for XGB
neg_f = int((y_train_f == 0).sum())
pos_f = int((y_train_f == 1).sum())
scale_pos_weight_f = neg_f / max(pos_f, 1)

neg_full = int((y_train_full == 0).sum())
pos_full = int((y_train_full == 1).sum())
scale_pos_weight_full = neg_full / max(pos_full, 1)

# 1) Logistic Regression (Fundamental)
log_fund = LogisticRegression(
    solver="saga",
    max_iter=600,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
log_fund.fit(X_train_fund_log_enc, y_train_f)

# 2) XGB (Fundamental)
xgb_fund = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight_f
)
xgb_fund.fit(X_train_fund_xgb_enc, y_train_f)

# 3) XGB (Full)
xgb_full = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight_full
)
xgb_full.fit(X_train_full_xgb_enc, y_train_full)

/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'aucpr'


In [ ]:
# Evaluation utilities + comparison table
def eval_scores(y_true, p):
    return {
        "roc_auc": roc_auc_score(y_true, p), # area under the receiver operating characteristic curve
        "pr_auc": average_precision_score(y_true, p), # precision and recall tradeoff area under curve
        "brier": brier_score_loss(y_true, p), # mean squared error of probabilistic predictions
    }

rows = []

# Logistic fundamental
p_val = log_fund.predict_proba(X_val_fund_log_enc)[:, 1]
p_test = log_fund.predict_proba(X_test_fund_log_enc)[:, 1]
val_m = eval_scores(y_val_f, p_val)
test_m = eval_scores(y_test_f, p_test)
rows.append({"model": "Logistic (Fundamental)", **{f"val_{k}": v for k, v in val_m.items()}, **{f"test_{k}": v for k, v in test_m.items()}})

# XGB fundamental
p_val = xgb_fund.predict_proba(X_val_fund_xgb_enc)[:, 1]
p_test = xgb_fund.predict_proba(X_test_fund_xgb_enc)[:, 1]
val_m = eval_scores(y_val_f, p_val)
test_m = eval_scores(y_test_f, p_test)
rows.append({"model": "XGB (Fundamental)", **{f"val_{k}": v for k, v in val_m.items()}, **{f"test_{k}": v for k, v in test_m.items()}})

# XGB full
p_val = xgb_full.predict_proba(X_val_full_xgb_enc)[:, 1]
p_test = xgb_full.predict_proba(X_test_full_xgb_enc)[:, 1]
val_m = eval_scores(y_val_full, p_val)
test_m = eval_scores(y_test_full, p_test)
rows.append({"model": "XGB (Full)", **{f"val_{k}": v for k, v in val_m.items()}, **{f"test_{k}": v for k, v in test_m.items()}})

summary = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False)
display(summary)

,model,val_roc_auc,val_pr_auc,val_brier,test_roc_auc,test_pr_auc,test_brier
2,XGB (Full),0.717833,0.216398,0.223962,0.703257,0.301722,0.227224
1,XGB (Fundamental),0.684061,0.190513,0.219695,0.688637,0.292369,0.213857
0,Logistic (Fundamental),0.667949,0.177110,0.237680,0.674078,0.272516,0.233573


In [8]:
# validation (F1) + test report
def best_threshold_by_f1(y_true, p):
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    # precision/recall has len = thresholds + 1
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    idx = int(np.argmax(f1_vals))
    return float(thresholds[idx]), float(f1_vals[idx])

# choose model to threshold (example: fundamental XGB)
p_val_fund = xgb_fund.predict_proba(X_val_fund_xgb_enc)[:, 1]
best_thr, best_f1 = best_threshold_by_f1(y_val_f, p_val_fund)
print("Best val threshold (F1):", round(best_thr, 4), "F1:", round(best_f1, 4))

p_test_fund = xgb_fund.predict_proba(X_test_fund_xgb_enc)[:, 1]
y_pred_test = (p_test_fund >= best_thr).astype(int)

print("Confusion matrix (test):")
print(confusion_matrix(y_test_f, y_pred_test))
print("\nClassification report (test):")
print(classification_report(y_test_f, y_pred_test, digits=4))

Best val threshold (F1): 0.5659 F1: 0.2636
Confusion matrix (test):
[[109129  32330]
 [ 14431  12831]]

Classification report (test):
              precision    recall  f1-score   support

           0     0.8832    0.7715    0.8236    141459
           1     0.2841    0.4707    0.3543     27262

    accuracy                         0.7229    168721
   macro avg     0.5837    0.6211    0.5889    168721
weighted avg     0.7864    0.7229    0.7477    168721

